# Week 4 — Extraction Exploration

This notebook validates the bronze extraction layer of the ETL pipeline.
We verify row counts, column names, and spot-check data quality for each bronze output.

## Setup

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
from src.orchestration import Pipeline
from src.utils import DataLoader

## Run Bronze Extraction

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

pipeline = Pipeline()
pipeline.run_bronze()

## 1. Certifications Bronze

**Expected:** ~20,221 rows (21,001 raw - 780 empty)

In [ ]:
cert = pipeline.bronze_certifications
print(f"Rows: {len(cert):,}")
print(f"Columns: {list(cert.columns)}")
print(f"\nNull counts:\n{cert.isnull().sum()}")
cert.head()

In [ ]:
# Verify no completely empty rows remain
fully_empty = cert.isnull().all(axis=1).sum()
print(f"Fully empty rows: {fully_empty} (should be 0)")
assert fully_empty == 0, "Empty rows should have been dropped!"

# Verify code uniqueness
print(f"Unique codes: {cert['code'].nunique():,}")
print(f"Duplicate codes: {cert['code'].duplicated().sum()}")

## 2. Clubs Bronze

**Expected:** 436 rows

In [ ]:
clubs = pipeline.bronze_clubs
print(f"Rows: {len(clubs):,}")
print(f"Columns: {list(clubs.columns)}")
print(f"\nDtypes:\n{clubs.dtypes}")
clubs.head()

In [ ]:
# Verify group_number is unique and integer
print(f"Unique group numbers: {clubs['group_number'].nunique()}")
print(f"group_number dtype: {clubs['group_number'].dtype}")
assert clubs['group_number'].nunique() == len(clubs), "group_number should be unique!"

## 3. Results Bronze

**Expected:** 112,375 total rows across 9 years

In [ ]:
results = pipeline.bronze_results
print(f"Total rows: {len(results):,}")
print(f"Columns: {list(results.columns)}")
print(f"\nRows per year:")
print(results['year'].value_counts().sort_index())

In [ ]:
# Verify expected per-year counts
expected = {
    2015: 14756, 2016: 12849, 2017: 14363, 2018: 13554,
    2019: 14009, 2022: 9409, 2023: 11798, 2024: 11239, 2025: 10398
}

actual = results['year'].value_counts().to_dict()
for year, exp in expected.items():
    act = actual.get(year, 0)
    status = "PASS" if act == exp else "FAIL"
    print(f"  {year}: {act:,} (expected {exp:,}) {status}")

In [ ]:
# Verify Summary (all) exists and is NaN for 2023
summary_null_by_year = results.groupby("year")["summary_all"].apply(lambda s: s.isna().all())
print("Summary (all) fully null per year:")
print(summary_null_by_year)
print(f"\n2023 summary all null: {summary_null_by_year.get(2023, False)} (should be True)")

## 4. Bronze Output Files

Verify CSVs were written to `data/bronze/`.

In [ ]:
from pathlib import Path

bronze_dir = Path("../../data/bronze")
for f in sorted(bronze_dir.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    df = pd.read_csv(f, nrows=0)
    print(f"{f.name:35s}  {size_kb:8.1f} KB  cols={len(df.columns)}")

## Summary

| Table | Raw Rows | Bronze Rows | Delta | Status |
|-------|----------|-------------|-------|--------|
| Certifications | 21,001 | 20,221 | -780 (empty rows) | PASS |
| Clubs | 436 | 436 | 0 | PASS |
| Results | 112,375 | 112,375 | 0 | PASS |

All bronze extractions pass validation. The pipeline is ready for the Silver (cleaning) layer in Week 5.